In [1]:
import pandas as pd
import re
import spacy
nlp = spacy.load("en_core_web_sm")

In [2]:
# Carregar os dois arquivos
df_train = pd.read_csv("../src/data/processed/df_train_processed.csv", index_col=0)

In [3]:
df_train = df_train.head(5000)

In [4]:
# Criar tabela de contagem por categoria
tabela_categorias = df_train["condition_label"].value_counts().reset_index()
tabela_categorias.columns = ["Categoria", "Quantidade"]

print(tabela_categorias)


   Categoria  Quantidade
0          5        1661
1          1        1111
2          4        1072
3          3         628
4          2         528


In [5]:
# modelo de baseline TF_IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tv = TfidfVectorizer(stop_words='english', ngram_range=(1,2), min_df=0.1, sublinear_tf=True)
tfidf = tv.fit_transform(df_train.medical_abstract_clean)
tfidf_df = pd.DataFrame(tfidf.toarray(), columns=tv.get_feature_names_out())
tfidf_df

,age,analysis,artery,associate,blood,case,cause,cell,change,clinical,...,suggest,surgery,therapy,time,treat,treatment,tumor,undergo,use,year
0,0.0,0.0,0.000000,0.220071,0.0,0.000000,0.0,0.481164,0.239693,0.205277,...,0.000000,0.254377,0.0,0.395503,0.0,0.0,0.000000,0.000000,0.000000,0.000000
1,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.258846,0.000000,0.000000,0.000000
2,0.0,0.0,0.000000,0.218582,0.0,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,0.427785,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000
3,0.0,0.0,0.000000,0.352434,0.0,0.000000,0.0,0.216861,0.000000,0.194160,...,0.208242,0.000000,0.0,0.000000,0.0,0.0,0.387554,0.000000,0.000000,0.000000
4,0.0,0.0,0.389886,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.220167,0.000000,0.188076
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.890399,0.000000
4996,0.0,0.0,0.502825,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.283943,0.000000,0.000000
4997,0.0,0.0,0.000000,0.000000,0.0,0.216008,0.0,0.264846,0.000000,0.237122,...,0.254320,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000
4998,0.0,0.0,0.000000,0.409583,0.0,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
import pandas as pd

# 1. Vetorização TF-IDF
tv = TfidfVectorizer(stop_words='english', ngram_range=(1,2), min_df=0.1, sublinear_tf=True)
X = tv.fit_transform(df_train['medical_abstract_clean'])

# 2. Labels
y = df_train['condition_name']

# 3. Separar treino e teste do SMOTE (evita data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 4. SMOTE no treino
smote = SMOTE(random_state=42, k_neighbors=1)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

# 5. Treinar modelo
rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=50,
    random_state=42
)   
rf_model.fit(X_resampled, y_resampled)

# 6. Predições 
y_pred = rf_model.predict(X_test)

# 7. Avaliação
print(classification_report(y_test, y_pred))

                                 precision    recall  f1-score   support

        cardiovascular diseases       0.49      0.47      0.48       214
      digestive system diseases       0.17      0.11      0.14       106
general pathological conditions       0.34      0.44      0.38       332
                      neoplasms       0.58      0.51      0.54       222
        nervous system diseases       0.20      0.15      0.17       126

                       accuracy                           0.39      1000
                      macro avg       0.36      0.34      0.34      1000
                   weighted avg       0.39      0.39      0.39      1000



In [7]:
# Função para normalizar texto
def lower_replace(text: str) -> str:
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)       # remove conteúdo entre colchetes
    text = re.sub(r'[^\w\s]', '', text)       # remove pontuação
    return text

# Tokenização + lematização + remoção de stopwords
def token_lemma_stop(text: str) -> list:
    doc = nlp(text)
    return [token.lemma_ for token in doc if not token.is_stop]

# Filtrar apenas certas classes gramaticais (exemplo: substantivos e adjetivos)
def filter_pos(tokens: list) -> str:
    doc = nlp(" ".join(tokens))
    return " ".join([token.text for token in doc if token.pos_ in ["NOUN", "ADJ", "PRON", "VERB"]])


# Pipeline único
def preprocess(text: str) -> list:
    text = lower_replace(text)
    tokens = token_lemma_stop(text)
    return filter_pos(tokens)


In [8]:
# Exemplo de novo texto
novo_texto = "Sexually transmitted diseases of the colon, rectum, and anus. The challenge of the nineties. During the past two decades, an explosive growth in both the prevalence and types of sexually transmitted diseases has occurred. Up to 55 percent of homosexual men with anorectal complaints have gonorrhea"
# Pré-processar
novo_texto_proc = preprocess(novo_texto)
print(novo_texto_proc)

# Vetorizar com o TF-IDF já treinado
X_novo = tv.transform([novo_texto_proc])
df = pd.DataFrame(X_novo.toarray(), columns=tv.get_feature_names_out())

display(df)

# Predição
pred = rf_model.predict(X_novo)
print("Classe prevista:", pred[0])


transmit disease colon rectum anus challenge past decade explosive growth prevalence type transmit disease occur percent homosexual man anorectal complaint gonorrhea


,age,analysis,artery,associate,blood,case,cause,cell,change,clinical,...,suggest,surgery,therapy,time,treat,treatment,tumor,undergo,use,year
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Classe prevista: neoplasms


In [9]:
probs = rf_model.predict_proba(X_novo)
print("Probabilidades por classe:", probs)


Probabilidades por classe: [[0.09082832 0.10049904 0.21217673 0.37787644 0.21861947]]
